In [1]:
!pip install transformers sentencepiece sacrebleu pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.1 MB/s eta 0:00:00


In [7]:
from pathlib import Path
from google.colab import drive
import pickle
import re
import time
import gc

import torch
import pandas as pd
import sacrebleu
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [8]:
drive.mount('/content/drive', force_remount=True)

dataset_path = Path("/content/drive/MyDrive/diplom/dataset")

with open("/content/drive/MyDrive/diplom/whisper_results_segments.pkl", "rb") as f:
    whisper_results = pickle.load(f)

print("Загружено видео:", len(whisper_results))
print("Ключи первого элемента:", whisper_results[0].keys())
print("Папки датасета:", [p.name for p in dataset_path.iterdir()])

Mounted at /content/drive
Загружено видео: 15
Ключи первого элемента: dict_keys(['folder', 'video', 'audio', 'ref', 'raw_text', 'segments', 'asr_time_sec'])
Папки датасета: ['video1', 'video2', 'video3', 'video4', 'video5', 'video6', 'video7', 'video8', 'video9', 'video10', 'video11', 'video12', 'video13', 'video14', 'video15']


Поиск эталонных русских переводов

In [9]:
def strip_srt(text: str) -> str:
    lines = text.splitlines()
    clean_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if line.isdigit():
            continue

        if "-->" in line:
            continue

        clean_lines.append(line)

    return " ".join(clean_lines)


def find_reference_ru(folder_name):
    folder_path = dataset_path / folder_name

    candidates = []

    for pattern in [
        "*reference*ru*.txt",
        "*ref*ru*.txt",
        "*ru*.txt",
        "*reference*.srt",
        "*ref*.srt",
        "*ru*.srt",
    ]:
        candidates.extend(folder_path.glob(pattern))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        return None, None

    ref_path = candidates[0]
    text = ref_path.read_text(encoding="utf-8")

    if ref_path.suffix.lower() == ".srt":
        text = strip_srt(text)

    return ref_path, text.strip()

In [10]:
references = {}

for item in whisper_results:
    folder = item["folder"]
    ref_path, ref_text = find_reference_ru(folder)

    if ref_text:
        references[folder] = ref_text
        print(folder, "->", ref_path.name)
    else:
        print("Не найден эталонный перевод:", folder)

print("Эталонных переводов найдено:", len(references))

video1 -> ru.srt
video10 -> ru.srt
video11 -> ru.srt
video12 -> ru.srt
video13 -> ru.srt
video14 -> ru.srt
video15 -> ru.srt
video2 -> ru.srt
video3 -> ru.srt
video4 -> ru.srt
video5 -> ru.srt
video6 -> ru.srt
video7 -> ru.srt
video8 -> ru.srt
video9 -> ru.srt
Эталонных переводов найдено: 15


Очистка текста

In [11]:
def clean_text_for_subs(text: str) -> str:
    text = re.sub(
        r"\b(\w+)([\s,]+\1\b)+",
        r"\1",
        text,
        flags=re.IGNORECASE,
    )

    fillers = r"\b(er|eh|hmm|mm-hmm|uh-huh|uh-uh|oh|ah|uh|huh|erm|um)\b[,\s]*"
    text = re.sub(fillers, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s{2,}", " ", text).strip()

    return text

Общие функции для моделей

In [12]:
DEVICE = "cpu"


def get_model_size_info(model):
    params = sum(p.numel() for p in model.parameters())
    size_mb_fp32 = params * 4 / 1024 / 1024

    return {
        "params_mln": round(params / 1_000_000, 1),
        "approx_size_mb_fp32": round(size_mb_fp32, 1),
    }

Opus-MT

In [13]:
def translate_segments_opus(
    segments,
    tokenizer,
    model,
    batch_size=8,
    max_new_tokens=128,
):
    texts_en_seg = [
        clean_text_for_subs(seg["text"])
        for seg in segments
    ]

    translated = []

    for i in range(0, len(texts_en_seg), batch_size):
        clean_texts = [
            t if t.strip() else "."
            for t in texts_en_seg[i:i + batch_size]
        ]

        batch = tokenizer(
            clean_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                num_beams=5,
                no_repeat_ngram_size=3,
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )

        translated.extend([d.strip() for d in decoded])

        del batch, generated
        gc.collect()

    ru_segments = []

    for seg, text_ru in zip(segments, translated):
        ru_segments.append({
            "start": seg["start"],
            "end": seg["end"],
            "text": text_ru,
        })

    return ru_segments

NLLB-600M

In [14]:
def translate_segments_nllb(
    segments,
    tokenizer,
    model,
    batch_size=2,
    max_new_tokens=128,
    src_lang="eng_Latn",
    tgt_lang="rus_Cyrl",
):
    tokenizer.src_lang = src_lang
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    texts_en_seg = [
        clean_text_for_subs(seg["text"])
        for seg in segments
    ]

    translated = []

    for i in range(0, len(texts_en_seg), batch_size):
        clean_texts = [
            t if t.strip() else "."
            for t in texts_en_seg[i:i + batch_size]
        ]

        batch = tokenizer(
            clean_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                num_beams=5,
                forced_bos_token_id=forced_bos_token_id,
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )

        translated.extend([d.strip() for d in decoded])

        del batch, generated
        gc.collect()

    ru_segments = []

    for seg, text_ru in zip(segments, translated):
        ru_segments.append({
            "start": seg["start"],
            "end": seg["end"],
            "text": text_ru,
        })

    return ru_segments

Основная функция эксперимента

In [15]:
all_results = []
translations_by_model = {}


def run_translation_experiment_for_model(
    model_label,
    model_name,
    model_type,
    src_lang=None,
    tgt_lang=None,
):
    print("=" * 70)
    print("Загрузка модели:", model_label)
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    size_info = get_model_size_info(model)

    model_translations = {}

    for idx, item in enumerate(tqdm(whisper_results), start=1):
        folder = item["folder"]
        segments = item["segments"]

        if folder not in references:
            continue

        print("\n" + "=" * 60)
        print(f"Видео {idx} из {len(whisper_results)}")
        print("Папка:", folder)
        print("Количество Whisper-сегментов:", len(segments))
        print("Запуск перевода...")

        start_time = time.time()

        if model_type == "opus":
            ru_segments = translate_segments_opus(
                segments=segments,
                tokenizer=tokenizer,
                model=model,
                batch_size=8,
                max_new_tokens=128,
            )

        elif model_type == "nllb":
            ru_segments = translate_segments_nllb(
                segments=segments,
                tokenizer=tokenizer,
                model=model,
                batch_size=2,
                max_new_tokens=128,
                src_lang=src_lang,
                tgt_lang=tgt_lang,
            )

        else:
            raise ValueError("Неизвестный тип модели")

        translation_time = time.time() - start_time

        print("Перевод завершён")
        print("Время перевода:", round(translation_time, 4), "сек")

        hypothesis = " ".join(seg["text"] for seg in ru_segments)
        reference = references[folder]

        bleu = sacrebleu.corpus_bleu(
            [hypothesis],
            [[reference]],
        ).score

        chrf = sacrebleu.corpus_chrf(
            [hypothesis],
            [[reference]],
        ).score

        print("BLEU:", round(bleu, 4))
        print("chrF:", round(chrf, 4))

        all_results.append({
            "folder": folder,
            "model": model_label,
            "BLEU": round(bleu, 4),
            "chrF": round(chrf, 4),
            "translation_time_sec": round(translation_time, 4),
            "num_whisper_segments": len(segments),
            "avg_time_per_segment_sec": round(
                translation_time / max(1, len(segments)),
                4,
            ),
            "params_mln": size_info["params_mln"],
            "approx_size_mb_fp32": size_info["approx_size_mb_fp32"],
        })

        model_translations[folder] = ru_segments

    translations_by_model[model_label] = model_translations

    del tokenizer
    del model
    gc.collect()

    print("Модель выгружена:", model_label)

Запуск Opus-MT

In [16]:
run_translation_experiment_for_model(
    model_label="Opus-MT",
    model_name="Helsinki-NLP/opus-mt-en-ru",
    model_type="opus",
)

Загрузка модели: Opus-MT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
  0%|          | 0/15 [00:00<?, ?it/s]


Видео 1 из 15
Папка: video1
Количество Whisper-сегментов: 198
Запуск перевода...


...

 93%|█████████▎| 14/15 [18:22<01:40, 100.16s/it]

Перевод завершён
Время перевода: 39.9457 сек
BLEU: 10.3237
chrF: 48.6441

Видео 15 из 15
Папка: video9
Количество Whisper-сегментов: 824
Запуск перевода...


100%|██████████| 15/15 [24:49<00:00, 99.32s/it] 

Перевод завершён
Время перевода: 386.6687 сек
BLEU: 19.7667
chrF: 62.7872


Модель выгружена: Opus-MT


Запуск NLLB-600M

In [17]:
run_translation_experiment_for_model(
    model_label="NLLB-600M",
    model_name="facebook/nllb-200-distilled-600M",
    model_type="nllb",
    src_lang="eng_Latn",
    tgt_lang="rus_Cyrl",
)

Загрузка модели: NLLB-600M


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

 93%|█████████▎| 14/15 [3:00:29<17:13, 1033.10s/it]

Перевод завершён
Время перевода: 429.6915 сек
BLEU: 9.7073
chrF: 48.9354

Видео 15 из 15
Папка: video9
Количество Whisper-сегментов: 824
Запуск перевода...
Перевод завершён
Время перевода: 3537.2542 сек


100%|██████████| 15/15 [3:59:26<00:00, 957.79s/it] 

BLEU: 17.38
chrF: 60.7546


Модель выгружена: NLLB-600M


Результаты

In [18]:
results_df = pd.DataFrame(all_results)
results_df

,folder,model,BLEU,chrF,translation_time_sec,num_whisper_segments,avg_time_per_segment_sec,params_mln,approx_size_mb_fp32
0,video1,Opus-MT,18.5480,61.0277,140.4312,198,0.7092,140.7,536.7
1,video10,Opus-MT,20.9017,51.1045,14.4284,26,0.5549,140.7,536.7
2,video11,Opus-MT,24.1810,59.7375,56.8029,37,1.5352,140.7,536.7
3,video12,Opus-MT,31.5570,58.0663,4.7881,10,0.4788,140.7,536.7
4,video13,Opus-MT,15.3926,47.4927,16.2541,18,0.9030,140.7,536.7
5,video14,Opus-MT,63.6665,77.9741,8.5626,11,0.7784,140.7,536.7
6,video15,Opus-MT,21.6893,56.0850,9.6201,18,0.5344,140.7,536.7
7,video2,Opus-MT,22.1687,52.9626,72.6471,111,0.6545,140.7,536.7
8,video3,Opus-MT,18.8544,59.4241,194.7406,308,0.6323,140.7,536.7
9,video4,Opus-MT,28.1562,69.7027,132.1057,165,0.8006,140.7,536.7


In [19]:
summary_df = (
    results_df
    .groupby("model", as_index=False)
    .agg({
        "BLEU": "mean",
        "chrF": "mean",
        "translation_time_sec": "mean",
        "avg_time_per_segment_sec": "mean",
        "params_mln": "first",
        "approx_size_mb_fp32": "first",
    })
    .sort_values(
        by=["chrF", "BLEU"],
        ascending=False,
    )
)

summary_df

,model,BLEU,chrF,translation_time_sec,avg_time_per_segment_sec,params_mln,approx_size_mb_fp32
1,Opus-MT,23.780180,59.038913,99.274673,0.709367,140.7,536.7
0,NLLB-600M,20.861353,57.747200,957.727460,6.877113,1402.1,5348.7


Сохранеие результатов

In [20]:
results_df.to_csv(
    "/content/drive/MyDrive/diplom/translation_model_results_correct.csv",
    index=False,
)

summary_df.to_csv(
    "/content/drive/MyDrive/diplom/translation_model_summary_correct.csv",
    index=False,
)